# Gymnasium and Atari: a practical overview

This notebook introduces the Gymnasium API by interacting with **Pong** through the Arcade Learning Environment (ALE). It covers:

- creating and resetting an Atari environment;
- inspecting observation and action spaces;
- taking actions and understanding the return values from `step`;
- collecting a simple random-agent episode;
- applying EfficientZero-style image preprocessing and frame stacking;
- running multiple environments; and
- avoiding common Atari integration mistakes.

No reinforcement-learning algorithm is used yet. The objective is to understand the environment boundary that EfficientZero will use.

## Running this notebook

The repository uses `uv`. From the repository root, synchronize dependencies with:

```bash
uv sync
```

Open this notebook in VS Code or another notebook interface and select the repository's `.venv` Python kernel. To launch a temporary browser-based JupyterLab without adding another project dependency, use:

```bash
uv run --with jupyterlab jupyter lab
```

Atari ROMs are bundled by current `ale-py` releases, so the old `AutoROM --accept-license` step found in older tutorials is not required.

In [ ]:
import cv2
import gymnasium as gym
import ale_py
import matplotlib.pyplot as plt
import numpy as np
import torch

print("Gymnasium:", gym.__version__)
print("ALE:", ale_py.__version__)
print("OpenCV:", cv2.__version__)
print("PyTorch:", torch.__version__)

## 1. Register the ALE environments

Gymnasium is the common environment API, while `ale-py` provides the Atari emulator and games. Explicit registration keeps the setup clear and works well with modern Gymnasium versions.

In [ ]:
gym.register_envs(ale_py)

atari_ids = sorted(
    spec.id for spec in gym.envs.registry.values() if spec.id.startswith("ALE/")
)
print(f"Registered {len(atari_ids)} Atari environments")
print("Examples:", atari_ids[:10])

Environment IDs such as `ALE/Pong-v5` identify a game and an environment version. The `v5` environment defaults include Atari-specific behavior such as frame skipping and sticky actions. Always save the exact ID and constructor options with an experiment.

## 2. Create Pong and inspect its interface

`render_mode` must be selected when the environment is created. `rgb_array` makes `render()` return an image and works in notebooks and headless training machines.

In [ ]:
ENV_ID = "ALE/Pong-v5"

env = gym.make(ENV_ID, render_mode="rgb_array")

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Render modes:", env.metadata.get("render_modes"))
print("Action meanings:", env.unwrapped.get_action_meanings())

For Pong, an observation is normally a `uint8` RGB image with shape `(210, 160, 3)`. The discrete action space contains integer action IDs. Action IDs are game-dependent, so inspect `get_action_meanings()` rather than hard-coding assumptions across games.

Accessing `env.unwrapped` reaches ALE-specific functionality beneath Gymnasium wrappers. Keep this usage isolated because it is less portable than the standard Gymnasium API.

## 3. Reset the environment

`reset()` starts a new episode and returns **two values**:

```python
observation, info = env.reset(seed=...)
```

The seed initializes this episode's environment random number generator. For reproducible experiments, also seed NumPy, PyTorch, action spaces, and every worker separately.

In [ ]:
observation, info = env.reset(seed=42)
env.action_space.seed(42)

print("Type:", type(observation))
print("Shape:", observation.shape)
print("Dtype:", observation.dtype)
print("Pixel range:", (observation.min(), observation.max()))
print("Info:", info)

plt.figure(figsize=(5, 6))
plt.imshow(observation)
plt.title("Initial Pong observation")
plt.axis("off")
plt.show()

## 4. Take one action

`step(action)` returns **five values**:

```python
next_observation, reward, terminated, truncated, info = env.step(action)
```

- `terminated`: the task reached a true terminal state.
- `truncated`: an external limit ended the episode, such as a time limit.
- The episode is over when `terminated or truncated` is true.

An agent must call `reset()` before stepping again after either condition. Keeping these flags separate is important when computing bootstrap value targets.

In [ ]:
action = env.action_space.sample()
next_observation, reward, terminated, truncated, info = env.step(action)

print("Action:", action, env.unwrapped.get_action_meanings()[action])
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
print("Observation shape:", next_observation.shape)
print("Info:", info)

Pong often needs the `FIRE` action to serve the ball. The helper below uses `FIRE` once and then behaves randomly. This is an environment demonstration, not a useful Pong policy.

In [ ]:
def run_random_episode(env: gym.Env, seed: int = 0, max_steps: int = 2_000):
    observation, info = env.reset(seed=seed)
    env.action_space.seed(seed)

    action_meanings = env.unwrapped.get_action_meanings()
    fire_action = action_meanings.index("FIRE") if "FIRE" in action_meanings else None

    total_reward = 0.0
    snapshots = [observation.copy()]

    for step in range(max_steps):
        action = fire_action if step == 0 and fire_action is not None else env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)

        if (step + 1) % 250 == 0:
            snapshots.append(observation.copy())

        if terminated or truncated:
            return total_reward, step + 1, True, snapshots, info

    return total_reward, max_steps, False, snapshots, info


total_reward, steps, episode_finished, snapshots, final_info = run_random_episode(
    env, seed=7
)
print(f"reward={total_reward}, steps={steps}, finished={episode_finished}")
print("final info:", final_info)

In [ ]:
columns = 4
rows = int(np.ceil(len(snapshots) / columns))
fig, axes = plt.subplots(rows, columns, figsize=(12, 3.5 * rows))
axes = np.asarray(axes).reshape(-1)

for index, axis in enumerate(axes):
    axis.axis("off")
    if index < len(snapshots):
        axis.imshow(snapshots[index])
        axis.set_title(f"Snapshot {index}")

plt.tight_layout()
plt.show()

### Optional visible window

On a desktop with display support, create a separate environment with `render_mode="human"` and step it in a loop. Do not use `human` rendering during training: it is slow and normally unavailable on headless Runpod machines. An environment's render mode cannot be changed after construction.

In [ ]:
# Set this to True only when running locally with a graphical display.
RUN_HUMAN_WINDOW = False

if RUN_HUMAN_WINDOW:
    human_env = gym.make(ENV_ID, render_mode="human")
    obs, info = human_env.reset(seed=0)
    for _ in range(1_000):
        obs, reward, terminated, truncated, info = human_env.step(
            human_env.action_space.sample()
        )
        if terminated or truncated:
            obs, info = human_env.reset()
    human_env.close()

## 5. EfficientZero-style preprocessing

The raw image is larger than the model needs. The original EfficientZero Atari setup uses resized RGB observations and four-frame context. Gymnasium provides reusable wrappers for this.

There is an important detail: `AtariPreprocessing` performs frame skipping itself, so the base ALE environment must be created with `frameskip=1`. Otherwise, frame skipping would be applied twice or Gymnasium would reject the configuration.

For a paper-oriented implementation, all choices—including screen size, RGB/grayscale, action repeat, sticky-action probability, reward clipping, no-op starts, and life-loss handling—must come from Hydra configuration and be verified against the pinned reference implementation.

In [ ]:
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation

# Close the earlier environment before creating another emulator instance.
env.close()

base_env = gym.make(
    ENV_ID,
    render_mode="rgb_array",
    frameskip=1,
    repeat_action_probability=0.0,
)
processed_env = AtariPreprocessing(
    base_env,
    frame_skip=4,
    screen_size=96,
    terminal_on_life_loss=False,
    grayscale_obs=False,
    scale_obs=False,
)
processed_env = FrameStackObservation(processed_env, stack_size=4)

stacked_observation, info = processed_env.reset(seed=42)
print("Stacked shape:", stacked_observation.shape)
print("Dtype:", stacked_observation.dtype)
print("Observation space:", processed_env.observation_space)

The wrapped observation shape is `(stack, height, width, channels)`, here `(4, 96, 96, 3)`. A convolutional PyTorch model normally expects channels first. Four RGB frames therefore become 12 input channels.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 4))
for frame_index, axis in enumerate(axes):
    axis.imshow(stacked_observation[frame_index])
    axis.set_title(f"Frame {frame_index}")
    axis.axis("off")
plt.tight_layout()
plt.show()

# (stack, height, width, RGB) -> (stack * RGB, height, width)
model_observation = (
    torch.from_numpy(stacked_observation)
    .permute(0, 3, 1, 2)
    .reshape(4 * 3, 96, 96)
    .float()
    .div(255.0)
)
print("Model tensor shape:", model_observation.shape)
print("Model tensor range:", (model_observation.min().item(), model_observation.max().item()))

During EfficientZero training, keep replay frames as `uint8` to reduce RAM usage. Convert, reorder, normalize, pin, and transfer sampled batches only at the model boundary. Before final implementation, verify whether the reference model uses `[0, 1]`, another normalization, or latent-state normalization; do not infer paper behavior from this introductory example.

## 6. Multiple environments

Gymnasium vector environments batch ordinary environments behind the same `reset`/`step` idea. This synchronous example is useful for learning the API. The final EfficientZero runtime will need separate self-play workers because every action requires MCTS and model inference.

In [ ]:
def make_pong_env():
    def thunk():
        return gym.make(ENV_ID)
    return thunk


vector_env = gym.vector.SyncVectorEnv([make_pong_env() for _ in range(2)])
vector_observations, vector_info = vector_env.reset(seed=123)
print("Batched observations:", vector_observations.shape)

actions = vector_env.action_space.sample()
result = vector_env.step(actions)
next_observations, rewards, terminated, truncated, info = result
print("Actions:", actions)
print("Rewards:", rewards)
print("Terminated:", terminated)
print("Truncated:", truncated)

vector_env.close()

## 7. Common mistakes

1. **Using the old four-value step API.** Modern Gymnasium returns five values from `step`.
2. **Ignoring truncation.** End an episode on `terminated or truncated`, but keep the distinction for value bootstrapping.
3. **Stepping after an episode ends.** Call `reset` first.
4. **Forgetting `FIRE`.** Some games remain idle until a fire/start action is sent.
5. **Applying frame skip twice.** Use `frameskip=1` in ALE when `AtariPreprocessing` handles skipping.
6. **Mixing training and evaluation semantics.** Reward clipping, sticky actions, life-loss handling, and exploration should be explicit.
7. **Storing float observations in replay.** Atari frames naturally fit in `uint8`; premature conversion greatly increases memory use.
8. **Assuming action IDs are universal.** Inspect each game's action meanings and action space.
9. **Not closing environments.** Call `close()` to release emulator and rendering resources.
10. **Comparing incompatible frame counts.** Report both agent transitions and raw emulator frames when action repeat is used.

## 8. A reusable interaction skeleton

Almost every Gymnasium agent eventually follows this control flow. EfficientZero will replace `action_space.sample()` with MCTS and will send each transition to replay.

In [ ]:
example_env = gym.make(ENV_ID)
observation, info = example_env.reset(seed=0)

for step in range(100):
    # EfficientZero will use: action = planner.search(observation, model)
    action = example_env.action_space.sample()

    next_observation, reward, terminated, truncated, info = example_env.step(action)

    # Replay will eventually store the observation, action, reward, search policy,
    # root value, done flags, and network version here.
    observation = next_observation

    if terminated or truncated:
        observation, info = example_env.reset()

example_env.close()
processed_env.close()
print("All environments closed.")

## Next exercises

1. Change `ENV_ID` to another registered Atari game and inspect how its actions differ.
2. Compare raw observations with grayscale `84×84` preprocessing.
3. Measure random-agent environment steps per second with and without rendering.
4. Add a wrapper that records separate agent-transition and raw-frame counters.
5. Implement a tiny replay deque that stores `(observation, action, reward, terminated, truncated)` as preparation for EfficientZero.

Useful documentation:

- [Gymnasium basic usage](https://gymnasium.farama.org/introduction/basic_usage/)
- [Gymnasium Atari environments](https://gymnasium.farama.org/environments/atari/)
- [AtariPreprocessing wrapper](https://gymnasium.farama.org/api/wrappers/misc_wrappers/#gymnasium.wrappers.AtariPreprocessing)
- [Gymnasium vector environments](https://gymnasium.farama.org/api/vector/)